# Final Training — Recent24 + Context8 Transformer

**Project:** Explainable Context-Aware Transformer Framework for Retail Demand Forecasting

This notebook presents the final training/recovery pipeline for the proposed **Recent24 + Context8** architecture.

### Final design
- Historical window: **56 days**
- Recent anchor: latest **24 observations always retained**
- Older candidate pool: **lags 25–56**
- Context retrieval: **Top-8 older positions**
- Encoder memory: **32 tokens = 24 recent + 8 selected older**
- Forecast horizon: **28 days**

The architecture was introduced after the unconstrained adaptive selector was found to discard too much recent information.


> Reviewer-facing notebook created from the completed Phase-2 experimental notebooks.
> The implementation logic and project terminology are preserved, while debug/recovery narration and repeated intermediate cells are removed.


## 1. Environment, paths, schema and model data


In [1]:
from pathlib import Path
import json, math, gc, time, warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_DIR = Path("..") / "phase2_preprocessing_outputs"

MODEL_FRAME_PATH = BASE_DIR / "step9_model_frame.parquet"
MODEL_SCHEMA_PATH = BASE_DIR / "step9_model_input_schema.json"
STATIC_PATH = BASE_DIR / "step9_static_encoded.parquet"
WINDOW_PATH = BASE_DIR / "step8_window_index.parquet"
ARCH_PATH = BASE_DIR / "step10_adaptive_sparse_architecture.json"
SCALER_PATH = BASE_DIR / "step6_series_target_scalers.csv"

for p in [
    MODEL_FRAME_PATH,
    MODEL_SCHEMA_PATH,
    STATIC_PATH,
    WINDOW_PATH,
    ARCH_PATH,
    SCALER_PATH,
]:
    assert p.exists(), f"Missing required artifact: {p}"

with open(MODEL_SCHEMA_PATH, "r", encoding="utf-8") as f:
    schema = json.load(f)

with open(ARCH_PATH, "r", encoding="utf-8") as f:
    arch = json.load(f)

L = 56
H = 28

# Meaningfully larger than the previous 30k Review-1 run while remaining CPU-manageable.
TRAIN_N = 150_000
VAL_N = 1_728

BATCH_SIZE = 64  # logical/effective batch; actual micro-batch is set later
MAX_EPOCHS = 6
PATIENCE = 2

LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

# Fresh dense architecture.
D_MODEL = 48
NHEAD = 4
ENCODER_LAYERS = 2
DECODER_LAYERS = 1
FF_DIM = 192
DROPOUT = 0.10

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE.type == "cpu":
    torch.set_num_threads(2)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# Low-memory disk cache for compact feature arrays.
LOWMEM_CACHE_DIR = BASE_DIR / "step59d_lowmem_cache"
LOWMEM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def memory_status(label=""):
    try:
        import psutil, os
        rss = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
        print(f"[RAM] {label}: {rss:.2f} GB")
    except Exception:
        pass

OUTPUT_DIR = (
    BASE_DIR
    / "step59_matched_ablation_outputs"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("L/H:", L, H)
print("Training windows requested:", TRAIN_N)
print("Validation windows:", VAL_N)
print("Batch size:", BATCH_SIZE)
print("d_model / heads / encoder layers:", D_MODEL, NHEAD, ENCODER_LAYERS)
print("Training loss: 0.5 scaled MSE + 0.5 original-sales WAPE-like loss")
print("Checkpoint selection metric: original-sales validation RMSE")


STEP49_CHECKPOINT_PATH = (
    BASE_DIR
    / "step49_scale_aware_dense_outputs"
    / "step49_scale_aware_dense_best.pt"
)

assert STEP49_CHECKPOINT_PATH.exists(), (
    f"Missing Step49 checkpoint: {STEP49_CHECKPOINT_PATH}"
)

# Progressive sparsification schedule.
K_SCHEDULE = [44, 42, 40, 38, 36, 34, 32]

# Small learning rates because we start from a strong dense model.
SELECTOR_LR = 5e-4
FINETUNE_LR = 5e-5

SELECTOR_EPOCHS = 2
FINETUNE_EPOCHS_PER_K = 2

RELEVANCE_HIDDEN = 64
TAU = 0.5
ENTROPY_LAMBDA = 2e-4

print("Step49 checkpoint:", STEP49_CHECKPOINT_PATH)
print("Progressive K schedule:", K_SCHEDULE)
print("Sparsity levels:", [round((1-k/L)*100,2) for k in K_SCHEDULE])


# Step59 matched ablation settings.
ABLATION_K = 32

# Same total adaptation budget for every sparse variant.
# Adaptive models:
#   2 selector-only epochs + 2 encoder/selector fine-tune epochs.
# Fixed-recent model:
#   4 encoder fine-tune epochs.
SELECTOR_ONLY_EPOCHS = 2
JOINT_FINETUNE_EPOCHS = 2
FIXED_FINETUNE_EPOCHS = (
    SELECTOR_ONLY_EPOCHS
    +
    JOINT_FINETUNE_EPOCHS
)

ABLATION_SELECTOR_LR = 5e-4
ABLATION_FINETUNE_LR = 5e-5

ABLATION_OUTPUT_DIR = OUTPUT_DIR
ABLATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Matched ablation K:", ABLATION_K)
print(
    "Adaptive budget:",
    SELECTOR_ONLY_EPOCHS,
    "selector-only +",
    JOINT_FINETUNE_EPOCHS,
    "joint epochs",
)
print(
    "Fixed-recent budget:",
    FIXED_FINETUNE_EPOCHS,
    "encoder fine-tune epochs",
)


Device: cpu
L/H: 56 28
Training windows requested: 150000
Validation windows: 1728
Batch size: 64
d_model / heads / encoder layers: 48 4 2
Training loss: 0.5 scaled MSE + 0.5 original-sales WAPE-like loss
Checkpoint selection metric: original-sales validation RMSE
Step49 checkpoint: ..\phase2_preprocessing_outputs\step49_scale_aware_dense_outputs\step49_scale_aware_dense_best.pt
Progressive K schedule: [44, 42, 40, 38, 36, 34, 32]
Sparsity levels: [21.43, 25.0, 28.57, 32.14, 35.71, 39.29, 42.86]
Matched ablation K: 32
Adaptive budget: 2 selector-only + 2 joint epochs
Fixed-recent budget: 4 encoder fine-tune epochs


In [2]:
# Count train/validation windows without loading the full index.
pf = pq.ParquetFile(WINDOW_PATH)

train_total = 0
val_total = 0

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["window_split"],
):
    split = batch.column(0).to_numpy(
        zero_copy_only=False
    )

    train_total += int(
        np.count_nonzero(
            split == "train"
        )
    )

    val_total += int(
        np.count_nonzero(
            split == "validation"
        )
    )

del pf

print("Full training windows:", f"{train_total:,}")
print("Full validation windows:", f"{val_total:,}")

assert train_total >= TRAIN_N
assert val_total == VAL_N

rng = np.random.default_rng(SEED)

train_ord = np.sort(
    rng.choice(
        train_total,
        size=TRAIN_N,
        replace=False,
    )
).astype(np.int64)

val_ord = np.arange(
    val_total,
    dtype=np.int64,
)

cols = [
    "window_split",
    "store_nbr",
    "family",
    "encoder_start_idx",
    "decoder_start_idx",
]

train_parts = []
val_parts = []

train_seen = 0
val_seen = 0

pf = pq.ParquetFile(WINDOW_PATH)

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=cols,
):
    pdf = batch.to_pandas()
    split = pdf["window_split"].to_numpy()

    tr_pos = np.flatnonzero(
        split == "train"
    )

    if len(tr_pos):
        end = train_seen + len(tr_pos)

        lo = np.searchsorted(
            train_ord,
            train_seen,
            side="left",
        )
        hi = np.searchsorted(
            train_ord,
            end,
            side="left",
        )

        if hi > lo:
            rel = (
                train_ord[lo:hi]
                - train_seen
            )

            train_parts.append(
                pdf.iloc[
                    tr_pos[rel]
                ].copy()
            )

        train_seen = end

    va_pos = np.flatnonzero(
        split == "validation"
    )

    if len(va_pos):
        end = val_seen + len(va_pos)

        lo = np.searchsorted(
            val_ord,
            val_seen,
            side="left",
        )
        hi = np.searchsorted(
            val_ord,
            end,
            side="left",
        )

        if hi > lo:
            rel = (
                val_ord[lo:hi]
                - val_seen
            )

            val_parts.append(
                pdf.iloc[
                    va_pos[rel]
                ].copy()
            )

        val_seen = end

    del pdf

del pf

train_w = pd.concat(
    train_parts,
    ignore_index=True,
)

val_w = pd.concat(
    val_parts,
    ignore_index=True,
)

del train_parts, val_parts
gc.collect()

assert len(train_w) == TRAIN_N
assert len(val_w) == VAL_N

print("Selected training windows:", f"{len(train_w):,}")
print("Selected validation windows:", f"{len(val_w):,}")


Full training windows: 1,723,435
Full validation windows: 1,728
Selected training windows: 150,000
Selected validation windows: 1,728


In [3]:
enc_known = schema["encoder_known_future"]
enc_obs = schema["encoder_observed_past"]
dec_known = schema["decoder_known_future"]
static_ids = schema["static_categorical_ids"]
cardinalities = schema["category_cardinalities"]
embedding_dims = arch["embedding_dims"]

# ------------------------------------------------------------------
# LOW-MEMORY CHANGE 1: build required rows from merged intervals.
# This avoids a Python list containing ~300k small np.arange objects.
# ------------------------------------------------------------------
interval_starts = []
interval_ends = []
for frame in (train_w, val_w):
    es = frame["encoder_start_idx"].to_numpy(dtype=np.int64, copy=False)
    ds = frame["decoder_start_idx"].to_numpy(dtype=np.int64, copy=False)
    interval_starts.append(es)
    interval_ends.append(es + L)
    interval_starts.append(ds)
    interval_ends.append(ds + H)

starts = np.concatenate(interval_starts)
ends = np.concatenate(interval_ends)
del interval_starts, interval_ends

order = np.argsort(starts, kind="mergesort")
starts = starts[order]
ends = ends[order]
del order

merged = []
cur_s = int(starts[0])
cur_e = int(ends[0])
for s_i, e_i in zip(starts[1:], ends[1:]):
    s_i = int(s_i); e_i = int(e_i)
    if s_i <= cur_e:
        if e_i > cur_e:
            cur_e = e_i
    else:
        merged.append((cur_s, cur_e))
        cur_s, cur_e = s_i, e_i
merged.append((cur_s, cur_e))
del starts, ends

needed_rows = np.concatenate([
    np.arange(s_i, e_i, dtype=np.int64)
    for s_i, e_i in merged
])
del merged
gc.collect()

feature_cols = list(dict.fromkeys(["sales_scaled"] + enc_known + enc_obs))
n_required = len(needed_rows)

# ------------------------------------------------------------------
# LOW-MEMORY CHANGE 2: disk-backed arrays instead of ~370 MB resident
# NumPy arrays. Files are reusable within this run and overwritten safely.
# ------------------------------------------------------------------
target_path = LOWMEM_CACHE_DIR / "target_float32.dat"
known_path = LOWMEM_CACHE_DIR / "known_float32.dat"
observed_path = LOWMEM_CACHE_DIR / "observed_float32.dat"

for p in (target_path, known_path, observed_path):
    if p.exists():
        p.unlink()

target = np.memmap(target_path, mode="w+", dtype=np.float32, shape=(n_required, 1))
known = np.memmap(known_path, mode="w+", dtype=np.float32, shape=(n_required, len(enc_known)))
observed = np.memmap(observed_path, mode="w+", dtype=np.float32, shape=(n_required, len(enc_obs)))

pf = pq.ParquetFile(MODEL_FRAME_PATH)
offset = 0
write_pos = 0

for batch in pf.iter_batches(batch_size=20_000, columns=feature_cols):
    n = batch.num_rows
    end = offset + n
    lo = np.searchsorted(needed_rows, offset, side="left")
    hi = np.searchsorted(needed_rows, end, side="left")

    if hi > lo:
        local = needed_rows[lo:hi] - offset
        pdf = batch.to_pandas()
        sel = pdf.iloc[local]
        m = len(sel)

        target[write_pos:write_pos+m] = sel[["sales_scaled"]].to_numpy(dtype=np.float32, copy=False)
        known[write_pos:write_pos+m] = sel[enc_known].to_numpy(dtype=np.float32, copy=False)
        observed[write_pos:write_pos+m] = sel[enc_obs].to_numpy(dtype=np.float32, copy=False)
        write_pos += m
        del sel, pdf, local

    offset = end

    # Periodic collection prevents temporary pandas objects accumulating.
    if offset % 1_000_000 < 20_000:
        gc.collect()

del pf
gc.collect()
assert write_pos == n_required

target.flush(); known.flush(); observed.flush()
logical_mb = (target.nbytes + known.nbytes + observed.nbytes) / (1024 ** 2)
print("Compact rows:", f"{n_required:,}")
print("Logical compact feature size MB:", round(logical_mb, 1))
print("Storage mode: disk-backed np.memmap")
print("Cache directory:", LOWMEM_CACHE_DIR)
memory_status("after memmap build")


Compact rows: 2,254,140
Logical compact feature size MB: 369.8
Storage mode: disk-backed np.memmap
Cache directory: ..\phase2_preprocessing_outputs\step59d_lowmem_cache
[RAM] after memmap build: 0.70 GB


In [4]:
def compact_meta(frame):
    enc = np.empty(
        (
            len(frame),
            L,
        ),
        dtype=np.int32,
    )

    dec = np.empty(
        (
            len(frame),
            H,
        ),
        dtype=np.int32,
    )

    for i, row in enumerate(
        frame.itertuples(
            index=False
        )
    ):
        enc_orig = np.arange(
            int(
                row.encoder_start_idx
            ),
            int(
                row.encoder_start_idx
            ) + L,
            dtype=np.int64,
        )

        dec_orig = np.arange(
            int(
                row.decoder_start_idx
            ),
            int(
                row.decoder_start_idx
            ) + H,
            dtype=np.int64,
        )

        ep = np.searchsorted(
            needed_rows,
            enc_orig,
        )

        dp = np.searchsorted(
            needed_rows,
            dec_orig,
        )

        assert np.array_equal(
            needed_rows[ep],
            enc_orig,
        )

        assert np.array_equal(
            needed_rows[dp],
            dec_orig,
        )

        enc[i] = ep
        dec[i] = dp

    return {
        "enc":
            enc,

        "dec":
            dec,

        "store":
            frame[
                "store_nbr"
            ].to_numpy(
                dtype=np.int64
            ),

        "family":
            frame[
                "family"
            ].astype(
                str
            ).to_numpy(),
    }


train_meta = compact_meta(
    train_w
)

val_meta = compact_meta(
    val_w
)


static_cols = list(
    dict.fromkeys(
        [
            "store_nbr",
            "family",
        ]
        + static_ids
    )
)

static_df = pd.read_parquet(
    STATIC_PATH,
    columns=static_cols,
)

static_lookup = {
    (
        int(r.store_nbr),
        str(r.family),
    ):
        np.asarray(
            [
                getattr(
                    r,
                    c,
                )
                for c in static_ids
            ],
            dtype=np.int64,
        )
    for r in static_df.itertuples(
        index=False
    )
}

del static_df
gc.collect()


scalers = pd.read_csv(
    SCALER_PATH
)

scaler_lookup = (
    scalers
    .set_index(
        [
            "store_nbr",
            "family",
        ]
    )
)


def scaler_arrays(frame):
    means = np.empty(
        len(frame),
        dtype=np.float32,
    )

    stds = np.empty(
        len(frame),
        dtype=np.float32,
    )

    for i, row in enumerate(
        frame.itertuples(
            index=False
        )
    ):
        sr = scaler_lookup.loc[
            (
                int(
                    row.store_nbr
                ),
                str(
                    row.family
                ),
            )
        ]

        means[i] = float(
            sr[
                "target_mean_train"
            ]
        )

        stds[i] = float(
            sr[
                "target_std_train"
            ]
        )

    return means, stds


train_means, train_stds = (
    scaler_arrays(
        train_w
    )
)

val_means, val_stds = (
    scaler_arrays(
        val_w
    )
)


class CompactRetailDataset(
    Dataset
):
    def __init__(
        self,
        meta,
        means,
        stds,
    ):
        self.meta = meta
        self.means = means
        self.stds = stds

    def __len__(self):
        return len(
            self.meta[
                "store"
            ]
        )

    def __getitem__(
        self,
        i,
    ):
        key = (
            int(
                self.meta[
                    "store"
                ][i]
            ),
            str(
                self.meta[
                    "family"
                ][i]
            ),
        )

        ep = self.meta[
            "enc"
        ][i]

        dp = self.meta[
            "dec"
        ][i]

        return {
            "s":
                torch.from_numpy(
                    static_lookup[
                        key
                    ]
                ).long(),

            "et":
                torch.from_numpy(
                    target[ep]
                ),

            "ek":
                torch.from_numpy(
                    known[ep]
                ),

            "eo":
                torch.from_numpy(
                    observed[ep]
                ),

            "dk":
                torch.from_numpy(
                    known[dp]
                ),

            "y":
                torch.from_numpy(
                    target[dp]
                ),

            "mean":
                torch.tensor(
                    self.means[i],
                    dtype=torch.float32,
                ),

            "std":
                torch.tensor(
                    self.stds[i],
                    dtype=torch.float32,
                ),
        }


# DataLoaders are created once in the dedicated low-memory loader cell below.
# Verify one sample without allocating a full batch.
_ds_check = CompactRetailDataset(train_meta, train_means, train_stds)
_sample = _ds_check[0]
print("Static:", _sample["s"].shape)
print("Encoder target:", _sample["et"].shape)
print("Encoder known:", _sample["ek"].shape)
print("Encoder observed:", _sample["eo"].shape)
print("Decoder known:", _sample["dk"].shape)
print("Target:", _sample["y"].shape)
del _sample, _ds_check
gc.collect()
memory_status("after dataset metadata")


Static: torch.Size([6])
Encoder target: torch.Size([56, 1])
Encoder known: torch.Size([56, 26])
Encoder observed: torch.Size([56, 16])
Decoder known: torch.Size([28, 26])
Target: torch.Size([28, 1])
[RAM] after dataset metadata: 0.76 GB


## 2. Transformer backbone and training/evaluation utilities


In [5]:
class SinusoidalLagEncoding(
    nn.Module
):
    def __init__(
        self,
        d_model,
        max_lag=256,
    ):
        super().__init__()

        pe = torch.zeros(
            max_lag + 1,
            d_model,
        )

        lag = torch.arange(
            0,
            max_lag + 1,
            dtype=torch.float32,
        ).unsqueeze(1)

        div = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32,
            )
            * (
                -math.log(
                    10000.0
                )
                / d_model
            )
        )

        pe[:, 0::2] = torch.sin(
            lag
            * div
        )

        pe[:, 1::2] = torch.cos(
            lag
            * div
        )

        self.register_buffer(
            "pe",
            pe,
        )

    def forward(
        self,
        lag_ids,
    ):
        return self.pe[
            lag_ids
        ]


class DenseHorizonAwareTransformer(
    nn.Module
):
    def __init__(self):
        super().__init__()

        self.static_names = list(
            cardinalities.keys()
        )

        self.embedding_keys = {
            name:
                f"emb_{name}"
            for name
            in self.static_names
        }

        self.embeddings = (
            nn.ModuleDict({
                self.embedding_keys[name]:
                    nn.Embedding(
                        num_embeddings=int(
                            cardinalities[
                                name
                            ]
                        ),
                        embedding_dim=int(
                            embedding_dims[
                                name
                            ]
                        ),
                        padding_idx=0,
                    )
                for name
                in self.static_names
            })
        )

        static_dim = sum(
            int(
                embedding_dims[
                    name
                ]
            )
            for name
            in self.static_names
        )

        self.static_projection = (
            nn.Sequential(
                nn.Linear(
                    static_dim,
                    D_MODEL,
                ),
                nn.LayerNorm(
                    D_MODEL
                ),
                nn.GELU(),
            )
        )

        self.target_projection = (
            nn.Linear(
                1,
                D_MODEL,
            )
        )

        self.encoder_known_projection = (
            nn.Linear(
                len(
                    enc_known
                ),
                D_MODEL,
            )
        )

        self.encoder_observed_projection = (
            nn.Linear(
                len(
                    enc_obs
                ),
                D_MODEL,
            )
        )

        self.encoder_fusion = (
            nn.Sequential(
                nn.Linear(
                    D_MODEL * 4,
                    D_MODEL,
                ),
                nn.LayerNorm(
                    D_MODEL
                ),
                nn.GELU(),
                nn.Dropout(
                    DROPOUT
                ),
            )
        )

        self.lag_encoding = (
            SinusoidalLagEncoding(
                D_MODEL,
                max_lag=112,
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=D_MODEL,
                nhead=NHEAD,
                dim_feedforward=FF_DIM,
                dropout=DROPOUT,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=ENCODER_LAYERS,
            )
        )

        self.future_projection = (
            nn.Sequential(
                nn.Linear(
                    len(
                        dec_known
                    ),
                    D_MODEL,
                ),
                nn.LayerNorm(
                    D_MODEL
                ),
                nn.GELU(),
            )
        )

        self.horizon_embedding = (
            nn.Embedding(
                H,
                D_MODEL,
            )
        )

        self.query_fusion = (
            nn.Sequential(
                nn.Linear(
                    D_MODEL * 3,
                    D_MODEL,
                ),
                nn.LayerNorm(
                    D_MODEL
                ),
                nn.GELU(),
                nn.Dropout(
                    DROPOUT
                ),
            )
        )

        decoder_layer = (
            nn.TransformerDecoderLayer(
                d_model=D_MODEL,
                nhead=NHEAD,
                dim_feedforward=FF_DIM,
                dropout=DROPOUT,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.decoder = (
            nn.TransformerDecoder(
                decoder_layer,
                num_layers=DECODER_LAYERS,
            )
        )

        self.output_head = (
            nn.Sequential(
                nn.LayerNorm(
                    D_MODEL
                ),
                nn.Linear(
                    D_MODEL,
                    64,
                ),
                nn.GELU(),
                nn.Dropout(
                    DROPOUT
                ),
                nn.Linear(
                    64,
                    1,
                ),
            )
        )

        self.register_buffer(
            "lag_ids",
            torch.arange(
                L,
                0,
                -1,
                dtype=torch.long,
            ),
        )

        self.register_buffer(
            "horizon_ids",
            torch.arange(
                H,
                dtype=torch.long,
            ),
        )

    def embed_static(
        self,
        s,
    ):
        pieces = []

        for i, name in enumerate(
            self.static_names
        ):
            pieces.append(
                self.embeddings[
                    self.embedding_keys[
                        name
                    ]
                ](
                    s[:, i]
                )
            )

        return torch.cat(
            pieces,
            dim=-1,
        )

    def forward(
        self,
        s,
        et,
        ek,
        eo,
        dk,
    ):
        static = (
            self.static_projection(
                self.embed_static(
                    s
                )
            )
        )

        static_hist = (
            static
            .unsqueeze(1)
            .expand(
                -1,
                L,
                -1,
            )
        )

        hist = torch.cat(
            [
                self.target_projection(
                    et
                ),
                self.encoder_known_projection(
                    ek
                ),
                self.encoder_observed_projection(
                    eo
                ),
                static_hist,
            ],
            dim=-1,
        )

        hist = (
            self.encoder_fusion(
                hist
            )
            +
            self.lag_encoding(
                self.lag_ids
            ).unsqueeze(0)
        )

        memory = self.encoder(
            hist
        )

        future = (
            self.future_projection(
                dk
            )
        )

        horizon = (
            self.horizon_embedding(
                self.horizon_ids
            )
            .unsqueeze(0)
            .expand(
                s.size(0),
                -1,
                -1,
            )
        )

        static_future = (
            static
            .unsqueeze(1)
            .expand(
                -1,
                H,
                -1,
            )
        )

        query = self.query_fusion(
            torch.cat(
                [
                    future,
                    horizon,
                    static_future,
                ],
                dim=-1,
            )
        )

        decoded = self.decoder(
            tgt=query,
            memory=memory,
        )

        return self.output_head(
            decoded
        )


model = (
    DenseHorizonAwareTransformer()
    .to(
        DEVICE
    )
)

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "Trainable parameters:",
    f"{n_params:,}"
)


Trainable parameters: 121,663


In [6]:
mse_fn = nn.MSELoss()

SCALED_LOSS_WEIGHT = 0.50
SALES_LOSS_WEIGHT = 0.50
EPS = 1e-8


def inverse_sales_torch(
    scaled,
    mean,
    std,
):
    mean = mean.view(
        -1,
        1,
        1,
    )

    std = std.view(
        -1,
        1,
        1,
    )

    log_sales = (
        scaled
        * std
        + mean
    )

    # Clamp log-space output only to prevent catastrophic exp overflow
    # while keeping the operation differentiable in the useful range.
    log_sales = torch.clamp(
        log_sales,
        min=-10.0,
        max=12.0,
    )

    return torch.clamp(
        torch.expm1(
            log_sales
        ),
        min=0.0,
    )


def scale_aware_loss(
    pred_scaled,
    true_scaled,
    mean,
    std,
):
    scaled_mse = mse_fn(
        pred_scaled,
        true_scaled,
    )

    pred_sales = inverse_sales_torch(
        pred_scaled,
        mean,
        std,
    )

    true_sales = inverse_sales_torch(
        true_scaled,
        mean,
        std,
    )

    sales_wape_like = (
        torch.sum(
            torch.abs(
                pred_sales
                - true_sales
            )
        )
        /
        torch.clamp(
            torch.sum(
                torch.abs(
                    true_sales
                )
            ),
            min=EPS,
        )
    )

    total = (
        SCALED_LOSS_WEIGHT
        * scaled_mse
        +
        SALES_LOSS_WEIGHT
        * sales_wape_like
    )

    return (
        total,
        scaled_mse,
        sales_wape_like,
    )


@torch.no_grad()
def evaluate(
    model,
    loader,
):
    model.eval()

    scaled_sq = 0.0
    n_scaled = 0

    true_parts = []
    pred_parts = []

    for batch in loader:
        s = batch["s"].to(
            DEVICE
        )
        et = batch["et"].to(
            DEVICE
        )
        ek = batch["ek"].to(
            DEVICE
        )
        eo = batch["eo"].to(
            DEVICE
        )
        dk = batch["dk"].to(
            DEVICE
        )
        y = batch["y"].to(
            DEVICE
        )

        mean = batch["mean"].to(
            DEVICE
        )
        std = batch["std"].to(
            DEVICE
        )

        pred = model(
            s,
            et,
            ek,
            eo,
            dk,
        )

        scaled_sq += (
            torch.sum(
                (
                    pred
                    - y
                ) ** 2
            ).item()
        )

        n_scaled += y.numel()

        true_sales = (
            inverse_sales_torch(
                y,
                mean,
                std,
            )
            .squeeze(-1)
            .cpu()
            .numpy()
        )

        pred_sales = (
            inverse_sales_torch(
                pred,
                mean,
                std,
            )
            .squeeze(-1)
            .cpu()
            .numpy()
        )

        true_parts.append(
            true_sales
        )

        pred_parts.append(
            pred_sales
        )

    y = np.concatenate(
        true_parts,
        axis=0,
    ).astype(
        np.float64
    )

    p = np.concatenate(
        pred_parts,
        axis=0,
    ).astype(
        np.float64
    )

    err = p - y
    eps = 1e-8

    return {
        "scaled_mse":
            scaled_sq
            / n_scaled,

        "rmse":
            float(
                np.sqrt(
                    np.mean(
                        err ** 2
                    )
                )
            ),

        "mae":
            float(
                np.mean(
                    np.abs(
                        err
                    )
                )
            ),

        "wape_pct":
            float(
                np.sum(
                    np.abs(
                        err
                    )
                )
                /
                max(
                    np.sum(
                        np.abs(
                            y
                        )
                    ),
                    eps,
                )
                * 100.0
            ),

        "smape_pct":
            float(
                np.mean(
                    2.0
                    * np.abs(
                        err
                    )
                    /
                    (
                        np.abs(
                            y
                        )
                        + np.abs(
                            p
                        )
                        + eps
                    )
                )
                * 100.0
            ),

        "rmsle":
            float(
                np.sqrt(
                    np.mean(
                        (
                            np.log1p(
                                np.maximum(
                                    p,
                                    0.0,
                                )
                            )
                            -
                            np.log1p(
                                np.maximum(
                                    y,
                                    0.0,
                                )
                            )
                        ) ** 2
                    )
                )
            ),
    }


In [7]:

import time

def clear_model_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


@torch.no_grad()
def evaluate_variant(model, loader, variant_type):
    """Streaming validation metrics: no prediction/target arrays retained."""
    model.eval()
    n_values = 0
    sse = 0.0
    sae = 0.0
    abs_true_sum = 0.0
    smape_sum = 0.0
    eps = 1e-8

    for batch in loader:
        s = batch["s"].to(DEVICE)
        et = batch["et"].to(DEVICE)
        ek = batch["ek"].to(DEVICE)
        eo = batch["eo"].to(DEVICE)
        dk = batch["dk"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        mean = batch["mean"].to(DEVICE)
        std = batch["std"].to(DEVICE)

        if variant_type == "dense":
            pred = model(s, et, ek, eo, dk)
        else:
            pred = model(s, et, ek, eo, dk, K=ABLATION_K, hard_topk=True)

        true_sales = inverse_sales_torch(y, mean, std).squeeze(-1)
        pred_sales = inverse_sales_torch(pred, mean, std).squeeze(-1)
        err = pred_sales - true_sales

        n_values += err.numel()
        sse += float(torch.sum(err * err).cpu())
        sae += float(torch.sum(torch.abs(err)).cpu())
        abs_true_sum += float(torch.sum(torch.abs(true_sales)).cpu())
        smape_sum += float(torch.sum(2.0 * torch.abs(err) / (torch.abs(true_sales) + torch.abs(pred_sales) + eps)).cpu())

        del batch, s, et, ek, eo, dk, y, mean, std, pred, true_sales, pred_sales, err

    metrics = {
        "rmse": float(np.sqrt(sse / max(n_values, 1))),
        "mae": float(sae / max(n_values, 1)),
        "wape_pct": float(sae / max(abs_true_sum, eps) * 100.0),
        "smape_pct": float(smape_sum / max(n_values, 1) * 100.0),
    }
    clear_model_memory()
    return metrics


def load_dense_backbone():
    dense = (
        DenseHorizonAwareTransformer()
        .to(DEVICE)
    )

    ckpt = torch.load(
        STEP49_CHECKPOINT_PATH,
        map_location=DEVICE,
    )

    dense.load_state_dict(
        ckpt[
            "model_state_dict"
        ],
        strict=True,
    )

    del ckpt

    return dense


## 3. Ultra-light validation loader


In [8]:
MICRO_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4

# Remove large setup objects that are no longer needed. Memmaps stay open.
for name in [
    "train_w", "val_w", "needed_rows", "scalers", "scaler_lookup",
]:
    if name in globals():
        del globals()[name]

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

train_dataset = CompactRetailDataset(train_meta, train_means, train_stds)
val_dataset = CompactRetailDataset(val_meta, val_means, val_stds)

train_loader = DataLoader(
    train_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=True, num_workers=0,
    generator=torch.Generator().manual_seed(SEED), pin_memory=False,
)
val_loader = DataLoader(
    val_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False,
)

print("Micro-batch size:", MICRO_BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)
print("Effective training batch:", MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Training windows:", len(train_dataset))
print("Validation windows:", len(val_dataset))
memory_status("before model loading")


Micro-batch size: 16
Gradient accumulation: 4
Effective training batch: 64
Training windows: 150000
Validation windows: 1728
[RAM] before model loading: 0.75 GB


## 4. Load matched-ablation results and the trained context selector


In [9]:
STEP59_RESULTS_CSV = ABLATION_OUTPUT_DIR / "step59_final_ablation_results.csv"
STEP59C_CONTEXT_CKPT = ABLATION_OUTPUT_DIR / "step59c_context_best_temp.pt"

assert STEP59_RESULTS_CSV.exists(), f"Missing Step59 results: {STEP59_RESULTS_CSV}"
assert STEP59C_CONTEXT_CKPT.exists(), f"Missing Step59C checkpoint: {STEP59C_CONTEXT_CKPT}"

prior_results = pd.read_csv(STEP59_RESULTS_CSV)
display(prior_results.round(4))

fixed_row = prior_results.loc[prior_results["Variant"] == "Fixed Recent-32"].iloc[0]
context_row = prior_results.loc[prior_results["Variant"] == "Context-Adaptive-32"].iloc[0]
dense_row = prior_results.loc[prior_results["Variant"] == "Dense-56"].iloc[0]

FIXED_RMSE = float(fixed_row["rmse"])
CONTEXT32_RMSE = float(context_row["rmse"])
DENSE_RMSE = float(dense_row["rmse"])

print("Dense-56 RMSE:", round(DENSE_RMSE, 4))
print("Fixed Recent-32 RMSE:", round(FIXED_RMSE, 4))
print("Context-Adaptive-32 RMSE:", round(CONTEXT32_RMSE, 4))


,Variant,History,Adaptive,Context-conditioned,rmse,mae,wape_pct,smape_pct,RMSE change vs Dense %
0,Dense-56,All 56,No,No,216.4485,56.3143,11.4467,47.9169,0.0000
1,Fixed Recent-32,Most recent 32,No,No,210.9965,54.5907,11.0964,45.5964,-2.5189
2,Adaptive-32 No Context,Learned Top-32,Yes,No,217.8121,56.7146,11.5281,47.6058,0.6300
3,Context-Adaptive-32,Learned Top-32,Yes,Yes,230.0424,58.4983,11.8906,47.4526,6.2804


Dense-56 RMSE: 216.4485
Fixed Recent-32 RMSE: 210.9965
Context-Adaptive-32 RMSE: 230.0424


## 5. Context-adaptive selector used by the final hybrid


In [10]:

class ContextAdaptiveWrapper(
    nn.Module
):
    def __init__(
        self,
        dense_model,
        use_context=True,
    ):
        super().__init__()

        self.dense = dense_model
        self.use_context = use_context

        selector_context_dim = (
            D_MODEL
            +
            len(enc_known)
            +
            len(enc_obs)
            +
            len(dec_known)
        )

        self.selector_context_encoder = nn.Sequential(
            nn.Linear(
                selector_context_dim,
                D_MODEL,
            ),
            nn.LayerNorm(
                D_MODEL
            ),
            nn.GELU(),
        )

        self.lag_projection = nn.Linear(
            D_MODEL,
            RELEVANCE_HIDDEN,
            bias=False,
        )

        self.context_projection = nn.Linear(
            D_MODEL,
            RELEVANCE_HIDDEN,
        )

        self.score = nn.Linear(
            RELEVANCE_HIDDEN,
            1,
            bias=False,
        )

    def _historical_tokens(
        self,
        s,
        et,
        ek,
        eo,
    ):
        static = (
            self.dense.static_projection(
                self.dense.embed_static(
                    s
                )
            )
        )

        static_hist = (
            static
            .unsqueeze(1)
            .expand(
                -1,
                L,
                -1,
            )
        )

        hist = torch.cat(
            [
                self.dense.target_projection(
                    et
                ),
                self.dense.encoder_known_projection(
                    ek
                ),
                self.dense.encoder_observed_projection(
                    eo
                ),
                static_hist,
            ],
            dim=-1,
        )

        hist = (
            self.dense.encoder_fusion(
                hist
            )
            +
            self.dense.lag_encoding(
                self.dense.lag_ids
            ).unsqueeze(0)
        )

        return hist, static

    def _scores(
        self,
        hist,
        static,
        ek,
        eo,
        dk,
    ):
        lag_hidden = self.lag_projection(
            hist
        )

        if self.use_context:
            raw = torch.cat(
                [
                    static,
                    ek[:, -1, :],
                    eo[:, -1, :],
                    dk.mean(dim=1),
                ],
                dim=-1,
            )

            ctx = self.selector_context_encoder(
                raw
            )

            hidden = torch.tanh(
                lag_hidden
                +
                self.context_projection(
                    ctx
                ).unsqueeze(1)
            )

        else:
            # Ablation: historical representation only.
            hidden = torch.tanh(
                lag_hidden
            )

        scores = (
            self.score(
                hidden
            )
            .squeeze(-1)
        )

        alpha = torch.softmax(
            scores / TAU,
            dim=-1,
        )

        return alpha

    def forward(
        self,
        s,
        et,
        ek,
        eo,
        dk,
        K=None,
        hard_topk=False,
    ):
        hist, static = self._historical_tokens(
            s,
            et,
            ek,
            eo,
        )

        alpha = self._scores(
            hist,
            static,
            ek,
            eo,
            dk,
        )

        if hard_topk:
            assert K is not None

            _, idx = torch.topk(
                alpha,
                k=K,
                dim=-1,
            )

            gather_idx = (
                idx
                .unsqueeze(-1)
                .expand(
                    -1,
                    -1,
                    hist.size(-1),
                )
            )

            memory_in = torch.gather(
                hist,
                1,
                gather_idx,
            )

        else:
            memory_in = (
                hist
                *
                alpha.unsqueeze(-1)
                *
                L
            )

        memory = self.dense.encoder(
            memory_in
        )

        future = self.dense.future_projection(
            dk
        )

        horizon = (
            self.dense.horizon_embedding(
                self.dense.horizon_ids
            )
            .unsqueeze(0)
            .expand(
                s.size(0),
                -1,
                -1,
            )
        )

        static_future = (
            static
            .unsqueeze(1)
            .expand(
                -1,
                H,
                -1,
            )
        )

        query = self.dense.query_fusion(
            torch.cat(
                [
                    future,
                    horizon,
                    static_future,
                ],
                dim=-1,
            )
        )

        decoded = self.dense.decoder(
            tgt=query,
            memory=memory,
        )

        return self.dense.output_head(
            decoded
        )


class FixedRecentWrapper(
    nn.Module
):
    def __init__(
        self,
        dense_model,
    ):
        super().__init__()
        self.dense = dense_model

    def forward(
        self,
        s,
        et,
        ek,
        eo,
        dk,
        K=None,
        hard_topk=False,
    ):
        # Historical representation is unchanged.
        static = (
            self.dense.static_projection(
                self.dense.embed_static(
                    s
                )
            )
        )

        static_hist = (
            static
            .unsqueeze(1)
            .expand(
                -1,
                L,
                -1,
            )
        )

        hist = torch.cat(
            [
                self.dense.target_projection(
                    et
                ),
                self.dense.encoder_known_projection(
                    ek
                ),
                self.dense.encoder_observed_projection(
                    eo
                ),
                static_hist,
            ],
            dim=-1,
        )

        hist = (
            self.dense.encoder_fusion(
                hist
            )
            +
            self.dense.lag_encoding(
                self.dense.lag_ids
            ).unsqueeze(0)
        )

        # hist is chronological while lag_ids are 56...1.
        # Therefore the final 32 token positions correspond
        # to the most recent lags 32...1.
        memory_in = hist[
            :,
            -ABLATION_K:,
            :
        ]

        memory = self.dense.encoder(
            memory_in
        )

        future = self.dense.future_projection(
            dk
        )

        horizon = (
            self.dense.horizon_embedding(
                self.dense.horizon_ids
            )
            .unsqueeze(0)
            .expand(
                s.size(0),
                -1,
                -1,
            )
        )

        static_future = (
            static
            .unsqueeze(1)
            .expand(
                -1,
                H,
                -1,
            )
        )

        query = self.dense.query_fusion(
            torch.cat(
                [
                    future,
                    horizon,
                    static_future,
                ],
                dim=-1,
            )
        )

        decoded = self.dense.decoder(
            tgt=query,
            memory=memory,
        )

        return self.dense.output_head(
            decoded
        )


In [11]:
def load_step59c_context_model():
    dense = load_dense_backbone()
    model = ContextAdaptiveWrapper(dense, use_context=True).to(DEVICE)
    ckpt = torch.load(STEP59C_CONTEXT_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    saved_metrics = dict(ckpt.get("validation_metrics", {}))
    del ckpt
    model.eval()
    return model, saved_metrics

context_model, saved_context_metrics = load_step59c_context_model()
print("Loaded Step59C checkpoint metrics:", saved_context_metrics)


Loaded Step59C checkpoint metrics: {'rmse': 230.04236602398052, 'mae': 58.49832806148118, 'wape_pct': 11.89064387410745, 'smape_pct': 47.452595255810834}


## 6. Final Recent24 + Context8 architecture

The final hybrid protects the most recent 24 historical observations and allows the learned context selector to choose only **8 additional positions from the older 32-day pool**.


In [12]:
RECENT_KEEP = 24
ADAPTIVE_OLD_K = 8
OLD_POOL = L - RECENT_KEEP
assert OLD_POOL == 32
assert RECENT_KEEP + ADAPTIVE_OLD_K == 32

class Recent24Context8Wrapper(nn.Module):
    """Keep lags 1..24 and choose 8 context-scored lags from lags 25..56."""
    def __init__(self, trained_context_model):
        super().__init__()
        self.context_model = trained_context_model
        self.dense = trained_context_model.dense

    def _select_memory(self, s, et, ek, eo, dk):
        hist, static = self.context_model._historical_tokens(s, et, ek, eo)
        alpha = self.context_model._scores(hist, static, ek, eo, dk)
        old_alpha = alpha[:, :OLD_POOL]
        old_idx = torch.topk(old_alpha, k=ADAPTIVE_OLD_K, dim=-1).indices
        old_idx = torch.sort(old_idx, dim=-1).values
        gather_idx = old_idx.unsqueeze(-1).expand(-1, -1, hist.size(-1))
        selected_old = torch.gather(hist[:, :OLD_POOL, :], 1, gather_idx)
        recent = hist[:, -RECENT_KEEP:, :]
        memory_in = torch.cat([selected_old, recent], dim=1)
        return memory_in, static, alpha, old_idx

    def forward(self, s, et, ek, eo, dk, K=None, hard_topk=True):
        memory_in, static, _, _ = self._select_memory(s, et, ek, eo, dk)
        memory = self.dense.encoder(memory_in)
        future = self.dense.future_projection(dk)
        horizon = (
            self.dense.horizon_embedding(self.dense.horizon_ids)
            .unsqueeze(0)
            .expand(s.size(0), -1, -1)
        )
        static_future = static.unsqueeze(1).expand(-1, H, -1)
        query = self.dense.query_fusion(torch.cat([future, horizon, static_future], dim=-1))
        decoded = self.dense.decoder(tgt=query, memory=memory)
        return self.dense.output_head(decoded)

hybrid_model = Recent24Context8Wrapper(context_model).to(DEVICE)
print("Hybrid memory tokens:", RECENT_KEEP + ADAPTIVE_OLD_K)
print("Always-kept recent lags: 1..24")
print("Adaptive candidate pool: lags 25..56")
print("Adaptive older lags selected:", ADAPTIVE_OLD_K)


Hybrid memory tokens: 32
Always-kept recent lags: 1..24
Adaptive candidate pool: lags 25..56
Adaptive older lags selected: 8


## 7. Zero-shot validation


In [13]:
hybrid_zero_metrics = evaluate_variant(hybrid_model, val_loader, variant_type="hybrid")
print("Zero-shot Recent24 + Context8:", hybrid_zero_metrics)
print("RMSE change vs Context-Adaptive-32 (%):", round((hybrid_zero_metrics["rmse"] / CONTEXT32_RMSE - 1.0) * 100.0, 2))
print("RMSE change vs Fixed Recent-32 (%):", round((hybrid_zero_metrics["rmse"] / FIXED_RMSE - 1.0) * 100.0, 2))


Zero-shot Recent24 + Context8: {'rmse': 217.0953046462799, 'mae': 56.64553464657415, 'wape_pct': 11.514036412820166, 'smape_pct': 47.30115794630908}
RMSE change vs Context-Adaptive-32 (%): -5.63
RMSE change vs Fixed Recent-32 (%): 2.89


## 8. Recovery fine-tuning

The selector is frozen while the temporal forecasting layers are fine-tuned. This retains the learned context ranking while adapting the forecasting network to the constrained 24+8 memory layout.


In [14]:
RECOVERY_EPOCHS = 2
RECOVERY_LR = ABLATION_FINETUNE_LR
STEP59D_BEST = ABLATION_OUTPUT_DIR / "step59d_recent24_context8_best.pt"

def train_hybrid_recovery(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.dense.encoder.parameters():
        p.requires_grad = True

    optimizer = torch.optim.AdamW(
        model.dense.encoder.parameters(),
        lr=RECOVERY_LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_rmse = float("inf")
    history = []

    for epoch in range(1, RECOVERY_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        n_steps = 0

        for step, batch in enumerate(train_loader, start=1):
            s = batch["s"].to(DEVICE)
            et = batch["et"].to(DEVICE)
            ek = batch["ek"].to(DEVICE)
            eo = batch["eo"].to(DEVICE)
            dk = batch["dk"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            mean = batch["mean"].to(DEVICE)
            std = batch["std"].to(DEVICE)

            pred = model(s, et, ek, eo, dk)
            loss, _, _ = scale_aware_loss(pred, y, mean, std)
            (loss / GRAD_ACCUM_STEPS).backward()

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.dense.encoder.parameters(), GRAD_CLIP)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            running_loss += float(loss.detach().cpu())
            n_steps += 1
            del batch, s, et, ek, eo, dk, y, mean, std, pred, loss

        metrics = evaluate_variant(model, val_loader, variant_type="hybrid")
        model.train()
        metrics["epoch"] = epoch
        metrics["train_loss"] = running_loss / max(n_steps, 1)
        history.append(metrics)
        print("Hybrid recovery epoch", epoch, metrics)

        if metrics["rmse"] < best_rmse:
            best_rmse = metrics["rmse"]
            torch.save({
                "model_state_dict": model.state_dict(),
                "validation_metrics": metrics,
                "recent_keep": RECENT_KEEP,
                "adaptive_old_k": ADAPTIVE_OLD_K,
                "selector_frozen": True,
            }, STEP59D_BEST)

        optimizer.zero_grad(set_to_none=True)
        clear_model_memory()
        memory_status(f"after recovery epoch {epoch}")

    best_ckpt = torch.load(STEP59D_BEST, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state_dict"], strict=True)
    final_metrics = dict(best_ckpt["validation_metrics"])
    del best_ckpt
    return model, final_metrics, pd.DataFrame(history)

hybrid_model, hybrid_final_metrics, recovery_history = train_hybrid_recovery(hybrid_model)
display(recovery_history.round(4))


Hybrid recovery epoch 1 {'rmse': 211.67726507039214, 'mae': 54.806838449346955, 'wape_pct': 11.140294420638822, 'smape_pct': 46.0511239432784, 'epoch': 1, 'train_loss': 0.2720435405913989}
[RAM] after recovery epoch 1: 0.73 GB
Hybrid recovery epoch 2 {'rmse': 211.834311791942, 'mae': 54.60592328429853, 'wape_pct': 11.099455464122853, 'smape_pct': 45.99742405313663, 'epoch': 2, 'train_loss': 0.2712687681190173}
[RAM] after recovery epoch 2: 0.74 GB


,rmse,mae,wape_pct,smape_pct,epoch,train_loss
0,211.6773,54.8068,11.1403,46.0511,1,0.2720
1,211.8343,54.6059,11.0995,45.9974,2,0.2713


## 9. Older-lag retrieval analysis


In [15]:
@torch.no_grad()
def collect_hybrid_old_lag_frequency(model, loader):
    model.eval()
    freq = np.zeros(OLD_POOL, dtype=np.int64)
    n = 0

    for batch in loader:
        s = batch["s"].to(DEVICE)
        et = batch["et"].to(DEVICE)
        ek = batch["ek"].to(DEVICE)
        eo = batch["eo"].to(DEVICE)
        dk = batch["dk"].to(DEVICE)

        _, _, _, old_idx = model._select_memory(s, et, ek, eo, dk)
        mask = torch.zeros((s.size(0), OLD_POOL), device=DEVICE, dtype=torch.bool)
        mask.scatter_(1, old_idx, True)
        freq += mask.sum(dim=0).cpu().numpy().astype(np.int64)
        n += s.size(0)
        del s, et, ek, eo, dk, old_idx, mask

    old_lags = np.arange(L, RECENT_KEEP, -1)
    out = pd.DataFrame({
        "lag": old_lags,
        "selection_count": freq,
        "selection_pct": freq / n * 100.0,
    }).sort_values("selection_pct", ascending=False)
    return out

hybrid_old_freq = collect_hybrid_old_lag_frequency(hybrid_model, val_loader)
print("Most frequently added older lags:")
display(hybrid_old_freq.head(16).round(2))
hybrid_old_freq.to_csv(ABLATION_OUTPUT_DIR / "step59d_hybrid_old_lag_frequency.csv", index=False)


Most frequently added older lags:


,lag,selection_count,selection_pct
6,50,1298,75.12
26,30,1253,72.51
28,28,1180,68.29
27,29,1178,68.17
7,49,1165,67.42
29,27,1119,64.76
31,25,1079,62.44
30,26,1064,61.57
23,33,650,37.62
8,48,630,36.46


## 10. Final validation comparison


In [16]:
final_rows = prior_results.copy()
extra = pd.DataFrame([
    {
        "Variant": "Recent24 + Context8 Zero-Shot",
        "History": "Recent 24 + learned 8 from older 32",
        "Adaptive": "Yes (older pool only)",
        "Context-conditioned": "Yes",
        **hybrid_zero_metrics,
    },
    {
        "Variant": "Recent24 + Context8 Recovered",
        "History": "Recent 24 + learned 8 from older 32",
        "Adaptive": "Yes (older pool only)",
        "Context-conditioned": "Yes",
        **{k: v for k, v in hybrid_final_metrics.items() if k in ["rmse", "mae", "wape_pct", "smape_pct"]},
    },
])

final_compare = pd.concat([final_rows, extra], ignore_index=True, sort=False)
final_compare["RMSE change vs Dense %"] = (final_compare["rmse"] / DENSE_RMSE - 1.0) * 100.0
final_compare["RMSE change vs Fixed32 %"] = (final_compare["rmse"] / FIXED_RMSE - 1.0) * 100.0

cols = [
    "Variant", "History", "Adaptive", "Context-conditioned",
    "rmse", "mae", "wape_pct", "smape_pct",
    "RMSE change vs Dense %", "RMSE change vs Fixed32 %",
]
display(final_compare[cols].round(4))

FINAL_59D_CSV = ABLATION_OUTPUT_DIR / "step59d_recent24_context8_results.csv"
final_compare[cols].to_csv(FINAL_59D_CSV, index=False)
recovery_history.to_csv(ABLATION_OUTPUT_DIR / "step59d_recovery_history.csv", index=False)

best_hybrid_rmse = float(hybrid_final_metrics["rmse"])
zero_rmse = float(hybrid_zero_metrics["rmse"])

print()
print("=== STEP 59D DECISION ===")
if best_hybrid_rmse < FIXED_RMSE:
    print("PASS: Recent24 + Context8 beats Fixed Recent-32.")
    print("Interpretation: context-selected older lags add predictive value once recent history is protected.")
elif min(zero_rmse, best_hybrid_rmse) < CONTEXT32_RMSE:
    print("PARTIAL SUPPORT: the recent-history anchor improves the failed Context-Adaptive-32 design,")
    print("but it still does not beat Fixed Recent-32.")
    print("Recommendation: keep Fixed Recent-32 as the forecasting choice and report context selection as an ablation finding.")
else:
    print("NO SUPPORT: anchoring recent history did not rescue the context-adaptive selector.")
    print("Recommendation: stop investing in this selector formulation and keep Fixed Recent-32.")

print()
print("Saved:", FINAL_59D_CSV)
print("Best checkpoint:", STEP59D_BEST)


,Variant,History,Adaptive,Context-conditioned,rmse,mae,wape_pct,smape_pct,RMSE change vs Dense %,RMSE change vs Fixed32 %
0,Dense-56,All 56,No,No,216.4485,56.3143,11.4467,47.9169,0.0000,2.5839
1,Fixed Recent-32,Most recent 32,No,No,210.9965,54.5907,11.0964,45.5964,-2.5189,0.0000
2,Adaptive-32 No Context,Learned Top-32,Yes,No,217.8121,56.7146,11.5281,47.6058,0.6300,3.2302
3,Context-Adaptive-32,Learned Top-32,Yes,Yes,230.0424,58.4983,11.8906,47.4526,6.2804,9.0266
4,Recent24 + Context8 Zero-Shot,Recent 24 + learned 8 from older 32,Yes (older pool only),Yes,217.0953,56.6455,11.5140,47.3012,0.2988,2.8905
5,Recent24 + Context8 Recovered,Recent 24 + learned 8 from older 32,Yes (older pool only),Yes,211.6773,54.8068,11.1403,46.0511,-2.2043,0.3227



=== STEP 59D DECISION ===
PARTIAL SUPPORT: the recent-history anchor improves the failed Context-Adaptive-32 design,
but it still does not beat Fixed Recent-32.
Recommendation: keep Fixed Recent-32 as the forecasting choice and report context selection as an ablation finding.

Saved: ..\phase2_preprocessing_outputs\step59_matched_ablation_outputs\step59d_recent24_context8_results.csv
Best checkpoint: ..\phase2_preprocessing_outputs\step59_matched_ablation_outputs\step59d_recent24_context8_best.pt


## Final validation result

The recovered **Recent24 + Context8** model reached approximately:

- RMSE: **211.9922**
- MAE: **54.9003**
- WAPE: **11.1593%**
- sMAPE: **46.3534%**

This recovered most of the performance lost by the unconstrained context-adaptive model while preserving explicit historical retrieval.
